# 2. VGG16 전이 학습 & ResNet vs VGG16 성능 비교

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

## 2-1. 데이터셋 준비

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2-2. VGG16 전이 학습 (Feature Extraction)

In [ ]:
vgg16 = models.vgg16(pretrained=True)

# Feature extraction: 사전 훈련된 가중치 동결
for param in vgg16.features.parameters():
    param.requires_grad = False

# 분류기 교체
vgg16.classifier[6] = nn.Linear(4096, 10)
vgg16 = vgg16.to(device)

print(vgg16.classifier)

## 2-3. VGG16 학습

In [ ]:
criterion = nn.CrossEntropyLoss()
vgg_optimizer = optim.Adam(filter(lambda p: p.requires_grad, vgg16.parameters()), lr=1e-4)

EPOCHS = 5
vgg_losses = []

for epoch in range(EPOCHS):
    vgg16.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        vgg_optimizer.zero_grad()
        outputs = vgg16(images)
        loss = criterion(outputs, labels)
        loss.backward()
        vgg_optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    vgg_losses.append(avg_loss)
    print(f'Epoch [{epoch+1}/{EPOCHS}] VGG16 Loss: {avg_loss:.4f}')

## 2-4. ResNet18 학습 (비교용)

In [ ]:
resnet = models.resnet18(pretrained=True)
resnet.fc = nn.Linear(resnet.fc.in_features, 10)
resnet = resnet.to(device)

resnet_optimizer = optim.Adam(resnet.parameters(), lr=1e-4)
resnet_losses = []

for epoch in range(EPOCHS):
    resnet.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        resnet_optimizer.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        resnet_optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    resnet_losses.append(avg_loss)
    print(f'Epoch [{epoch+1}/{EPOCHS}] ResNet18 Loss: {avg_loss:.4f}')

## 2-5. 성능 비교

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total * 100

vgg_acc    = evaluate(vgg16,  test_loader)
resnet_acc = evaluate(resnet, test_loader)

print(f'VGG16   Test Accuracy: {vgg_acc:.2f}%')
print(f'ResNet18 Test Accuracy: {resnet_acc:.2f}%')

# Loss 곡선 비교
epochs = range(1, EPOCHS+1)
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, vgg_losses, marker='o', label='VGG16')
plt.plot(epochs, resnet_losses, marker='s', label='ResNet18')
plt.title('Training Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.bar(['VGG16', 'ResNet18'], [vgg_acc, resnet_acc], color=['steelblue', 'tomato'])
plt.title('Test Accuracy Comparison')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)

plt.tight_layout()
plt.show()